# Checkpoint — Recommender systems

Dit notebook wordt automatisch gecheckt via pytest (GitHub Actions). De checks voeren je notebook uit en valideren of bepaalde variabelen bestaan en binnen de verwachte ranges vallen. **Je krijgt geen hints in de foutmeldingen: het is aan jou om correct werkende code te schrijven.**

## BELANGRIJK: gebruik exact de genoemde variabelennamen

De checks zoeken volgende variabelen in je notebook. Andere namen worden niet gezien:

- `user_item_matrix` — de user-item-matrix uit de speelgoed-data (pandas DataFrame, rijen = users, kolommen = songs, ontbrekende ratings = 0)
- `sparse_user_item_matrix` — dezelfde matrix in sparse **CSR**-formaat (`scipy.sparse.csr_matrix`)
- `als_toy_model` — een **getraind** `AlternatingLeastSquares`-model op de speelgoed-matrix
- `antwoord_feedback` — meerkeuzevraag 1: `"A"`, `"B"`, `"C"` of `"D"`
- `antwoord_orientatie` — meerkeuzevraag 2: `"A"`, `"B"`, `"C"` of `"D"`

Gebruik voor het speelgoed-model `AlternatingLeastSquares(factors=50, random_state=0)` en train het met `als_toy_model.fit(sparse_user_item_matrix)`.


## Oefening 1 — Speelgoed-data: pivot-tabel en sparse-matrix

We vertrekken van een kleine dataset met ratings (`user_id`, `song_id`, `rating`). Zet die om naar een **user-item-matrix** met `df.pivot_table(...)` (rijen = users, kolommen = songs, ontbrekende ratings = 0) en daarna naar een **sparse CSR-matrix** met `scipy.sparse.csr_matrix`. Ten slotte train je een `AlternatingLeastSquares`-model op die sparse matrix.


In [10]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

data = {
    'user_id': [1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6],
    'song_id': [1, 2, 1, 3, 4, 5, 5, 3, 5, 2, 4, 6],
    'rating': [4, 3, 5, 2, 4, 1, 3, 5, 2, 4, 3, 5]
}

df = pd.DataFrame(data)


In [8]:
# TODO: zet df om in een user-item-matrix met behulp van een pivot-tabel.
# Rijen = user_id, kolommen = song_id, values = rating, ontbrekende waarden = 0.
user_item_matrix = df.pivot_table(values='rating', index='user_id', columns='song_id', fill_value=0) # vervang door df.pivot_table(...)
user_item_matrix


song_id,1,2,3,4,5,6
user_id,,,,,,
1,4.0,3.0,0.0,0.0,0.0,0.0
2,5.0,0.0,2.0,0.0,0.0,0.0
3,0.0,0.0,0.0,4.0,2.0,0.0
4,0.0,0.0,5.0,0.0,2.0,0.0
5,0.0,4.0,0.0,3.0,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,5.0


In [9]:
# TODO: zet user_item_matrix om naar een sparse matrix in CSR-formaat.
sparse_user_item_matrix = csr_matrix(user_item_matrix.values) # vervang door csr_matrix(user_item_matrix.values)
sparse_user_item_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (6, 6)>

In [15]:
# TODO: maak een AlternatingLeastSquares-model met factors=50 en random_state=0,
# train het op sparse_user_item_matrix en bewaar het model in als_toy_model.
als_toy_model = AlternatingLeastSquares(factors=50, random_state=0)  # vervang door een getraind model
als_toy_model.fit(sparse_user_item_matrix)
als_toy_model

100%|██████████| 15/15 [00:00<00:00, 72.12it/s]


In [16]:
# Optioneel (NIET gecheckt): bekijk de aanbevelingen voor user 1.
ids_toy, scores_toy = als_toy_model.recommend(1, sparse_user_item_matrix[1], N=3)
print(ids_toy, scores_toy)


[1 4 5] [ 3.6003003e-03  2.1806508e-03 -5.9623271e-06]


## Oefening 2 — last.fm 360K (optionele uitbreiding, NIET gecheckt)

Deze uitbreiding wordt **niet** automatisch gecheckt. Je kan haar doen voor verdieping, precies zoals in `recommender_systems_oefeningen.ipynb`:

- Download de data met `from implicit.datasets.lastfm import get_lastfm`.
- Weeg de `(item, user)`-matrix met `bm25_weight` uit `implicit.nearest_neighbours` en transponeer daarna naar een `(user, item)` CSR-matrix `user_plays`.
- Train `AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)` en maak aanbevelingen met `.recommend`.

Deze uitbreiding vereist dat `implicit` (en `h5py`) geïnstalleerd zijn via `uv sync`.


In [32]:
from implicit.als import AlternatingLeastSquares
from implicit.datasets.lastfm import get_lastfm
from implicit.nearest_neighbours import bm25_weight

# 1. Unpack the three values returned by get_lastfm()
artists, users, plays = get_lastfm()

# 2. Transpose to user-item format and pass it into bm25_weight()
# bm25_weight already returns a sparse matrix (coo_matrix), so convert directly to csr
user_plays = bm25_weight(plays.T, K1=100, B=0.8).tocsr()

# 3. Fit the model
model = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=2.0)
model.fit(user_plays)

# 4. Get recommendations (returns item IDs and their corresponding scores)
userid = 0
ids, scores = model.recommend(userid, user_plays[userid])

# Print the recommended artist names and scores
for artist_id, score in zip(ids, scores):
    print(f"{artists[artist_id]}: {score:.4f}")

100%|██████████| 15/15 [09:53<00:00, 39.56s/it]

kittie: 1.3585
blackmore's night: 1.3151
in this moment: 1.2991
juliette and the licks: 1.2863
doro: 1.2705
emilie autumn: 1.2650
flyleaf: 1.2445
the donnas: 1.2284
exilia: 1.2192
die happy: 1.2182


In [ ]:
# 1. Search for an artist by name (e.g., "Radiohead")
search_query = ""
artist_id = 292302

# Loop through the artists array to find the matching ID
for idx, artist_name in enumerate(artists):
    # Depending on the dataset, artist_name might be bytes, so decode if necessary
    name = artist_name.decode("utf-8") if isinstance(artist_name, bytes) else str(artist_name)
    if search_query in name.lower():
        artist_id = idx
        print(f"Found '{name}' at index ID: {artist_id}")
        break

if artist_id is not None:
    # 2. Test 1: Find similar artists using model.similar_items()
    similar_ids, similar_scores = model.similar_items(artist_id, N=10)
    
    print(f"\nArtists most similar to '{artists[artist_id]}':")
    for sim_id, score in zip(similar_ids, similar_scores):
        print(f"  - {artists[sim_id]} (Score: {score:.4f})")
else:
    print(f"Artist '{search_query}' not found in the dataset.")

TypeError: 'in <string>' requires string as left operand, not NoneType

In [42]:
for id, artist in enumerate(artists):
    print(id, artist)

0  2 
1  58725ab=>
2  80lİ yillarin tÜrkÇe sÖzlÜ aŞk Şarkilari
3  amy winehouse
4  cours de la somme
5  fatboy slim
6  kanye west
7  mala rodriguez
8  mohamed lamine
9  oliver shanti & friends
10  the ranting gryphon
11 ! '
12 ! !"+
13 ! bruno fergani / ftp !
14 ! music for donations !
15 ! www.polskie-mp3.tk ! breakout
16 ! www.polskie-mp3.tk ! czan
17 ! www.polskie-mp3.tk ! lista przebojow programu iii
18 ! www.polskie-mp3.tk ! mr. zoob
19 ! Àëåêñàíäð Ðûáàê
20 !! dj fergie !!
21 !!!
22 !!! (chk chk chk )
23 !!! (chk chk chk)
24 !!! (chk,chk,chk)
25 !!!!!!
26 !"
27 !0<>5 >;lh>5 @>ab>5 '8a;>
28 !525@=k5 @0b0
29 !5:b>@ 3070
30 !5@359 01:8=
31 !5@359 @8ab>2a:89
32 !5@51@>
33 !5@530
34 !5@f5289 0?04
35 !:0d0=4@
36 !;>b
37 !=56=>
38 !?;8=
39 !action pact!
40 !attention!
41 !b0a l5e0
42 !b8<#
43 !bang elektronika
44 !christopher:christopher
45 !cube
46 !d!
47 !deladap
48 !distain
49 !exclusive! dj humo
50 !green day
51 !hero
52 !jazzattakk!
53 !k7
54 !k7®
55 !naamloos?
56 !n

In [ ]:
userid = 
ids, scores = model.recommend(userid, user_plays[userid])

# Print the recommended artist names and scores
for artist_id, score in zip(ids, scores):
    print(f"{artists[artist_id]}: {score:.4f}")
for id, artist in enumerate(artists):
    if id == userid:
        print(artist)

the sounds: 1.1174
the ting tings: 1.0681
enter shikari: 1.0654
arctic monkeys: 1.0444
the kooks: 1.0438
the killers: 1.0437
fall out boy: 1.0422
kill hannah: 1.0389
from first to last: 1.0357
bloc party: 1.0341
metallica


## Meerkeuzevragen

### Vraag 1 — Impliciete vs. expliciete feedback
In een recommender-systeem met **impliciete feedback** tellen we hoe vaak een gebruiker met een item interageert (bv. hoe vaak iemand een nummer afspeelt). Welke van de volgende is een **voorbeeld van expliciete feedback**?

- **A)** Het aantal keren dat een gebruiker een nummer heeft afgespeeld.
- **B)** Een gebruiker geeft een rating van 5 sterren aan een film.
- **C)** Het aantal keren dat een gebruiker een pagina heeft bekeken.
- **D)** Een gebruiker klikt op een aanbevolen (onbekend) nummer en luistert het volledig af.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_feedback`.

### Vraag 2 — Oriëntatie van de user-item-matrix
De `implicit`-bibliotheek verwacht voor ALS dat de ingevoerde matrix **users als rijen** en **items als kolommen** heeft. De `get_lastfm()`-functie geeft echter een matrix terug waarvan elke rij een artiest is en elke kolom een gebruiker. Wat moet je doen om dit te corrigeren?

- **A)** Niets; de matrix werkt al correct.
- **B)** De matrix transponeren met `.T` en omzetten naar CSR-formaat.
- **C)** De matrix normaliseren met `StandardScaler`.
- **D)** Het model trainen op de originele matrix (rijen = artiesten).

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_orientatie`.


In [ ]:
# Vraag 1: antwoord_feedback = "A", "B", "C" of "D"
antwoord_feedback = ""

# Vraag 2: antwoord_orientatie = "A", "B", "C" of "D"
antwoord_orientatie = "B"
